L'idée : C'est que le client fournit un audio( vocale) le chiffre en cryptage fully homomrophique , et l'envoie au serveur  :

Le serveur ferra deux manipulations intéressantes:
- calcule la durée du silence de l'audio  et la durée du son  
- comparer deux audios pour savoir si deux personnes ont dit la même chose ( plus difficile mais théoriqument faisable)
  |-> Comme pour faire ceci il suffit de transformé les audios en vecteurs , les chiffrées  puis le serveur calcule la distance euclidienne ou la corrélation
  |-> Limite comme c'est principalement des vecteur de flottants , on va devoir manipuler avec un schéma CKKS  , sauf que genère du bruit après chaque multiplication,
  donc va falloir bien manipuler le contexte et choisir ses arguments 


In [2]:
import scipy as sp
import numpy as np
import librosa
import tenseal as ts
import utils
import soundfile as sf


In [ ]:

audio_data = {}
sample_rates = {}

files_to_load = {
    "artificiel_pomme_java": 'my_data/artificiel_pomme_java.mp3',
    "python_java_cetic_1s_celine": 'my_data/python_java_cetic_1s_celine_voice.mp3',
    "python_java_cetic_1s_mathieu": 'my_data/python_java_cetic_1s_mathieu_voice.mp3',
    "python_java_cetic_nodelay_celine": 'my_data/python_java_cetic_nodelay_celine_voice.mp3',
    "python_java_cetic_nodelay_mathieu": 'my_data/python_java_cetic_nodeylay_mathieu_voice.mp3'
}

for name, path in files_to_load.items():
    try:
        data, sr = librosa.load(path, sr=None)
        audio_data[name] = data
        sample_rates[name] = sr
        print(f" {name} chargé: {len(data)} échantillons à {sr}Hz")
    except Exception as e:
        print(f" Erreur avec {path}: {str(e)}")
        audio_data[name] = None

max_length = max(len(data) for data in audio_data.values() if data is not None)
print(f"\n🔈 Longueur maximale: {max_length} échantillons")

def pad_audio(data, target_length):
    if len(data) < target_length:
        padding = target_length - len(data)
        fade_out = np.linspace(1, 0, min(500, len(data)))  
        data[-500:] = data[-500:] * fade_out
        return np.pad(data, (0, padding), 'constant')
    return data[:target_length]

processed_audios = {}
for name, data in audio_data.items():
    if data is not None:
        padded = pad_audio(data, max_length)
        amplified = padded * 1000
        processed_audios[name] = amplified
        print(f" {name} traité: {len(amplified)} échantillons (padding: {max_length-len(data) if len(data)<max_length else 0})")

context = ts.context(
    ts.SCHEME_TYPE.CKKS,
    poly_modulus_degree=8192,
    coeff_mod_bit_sizes=[60, 40, 40, 60]
)
context.generate_galois_keys()
context.global_scale = 2 ** 40

my_secret_key = context.serialize(save_secret_key=True)
utils.write_data("Keys_provider/secret_key.txt", my_secret_key)
context.make_context_public()
public_key = context.serialize()
utils.write_data("Keys_provider/public_key.txt", public_key)

for name, data in processed_audios.items():
    encrypted = ts.ckks_vector(context, data)
    output_path = f"output_provider/{name}_encrypted.txt"
    utils.write_data(output_path, encrypted.serialize())
    print(f" {name} chiffré → {output_path}")

print("\n Traitement terminé avec succès!")
print(f"• Clés dans Keys_provider/")
print(f"• Audios chiffrés dans output_provider/")


C:\Users\Lambika\AppData\Local\Temp\ipykernel_4184\3598970780.py:14: UserWarning: PySoundFile failed. Trying audioread instead.
  data, sr = librosa.load(path, sr=None)


 Erreur avec my_data/artificiel_pomme_java.mp3: [Errno 2] No such file or directory: 'my_data/artificiel_pomme_java.mp3'
 python_java_cetic_1s_celine chargé: 98496 échantillons à 24000Hz
 python_java_cetic_1s_mathieu chargé: 100800 échantillons à 24000Hz
 python_java_cetic_nodelay_celine chargé: 31680 échantillons à 24000Hz
 python_java_cetic_nodelay_mathieu chargé: 33408 échantillons à 24000Hz

🔈 Longueur maximale: 100800 échantillons
 python_java_cetic_1s_celine traité: 100800 échantillons (padding: 2304)
 python_java_cetic_1s_mathieu traité: 100800 échantillons (padding: 0)
 python_java_cetic_nodelay_celine traité: 100800 échantillons (padding: 69120)
 python_java_cetic_nodelay_mathieu traité: 100800 échantillons (padding: 67392)
The following operations are disabled in this setup: matmul, matmul_plain, enc_matmul_plain, conv2d_im2col.
If you need to use those operations, try increasing the poly_modulus parameter, to fit your input.
 python_java_cetic_1s_celine chiffré → output_prov

In [1]:
# secret_key_data = utils.read_data("Keys_provider/secret_key.txt")
# context = ts.context_from(secret_key_data)
# import math
# if not context.is_private():
#     raise ValueError("Clé secrète manquante pour le déchiffrement.")

# diff_data = utils.read_data("outputs_manipulator/difference.txt")
# diff_vec = ts.ckks_vector_from(context, diff_data)
# diff_decrypted = diff_vec.decrypt()
# distance_squared = sum(diff_decrypted)
# distance = math.sqrt(distance_squared)

# dot_data = utils.read_data("outputs_manipulator/dot_product.txt")
# dot_vec = ts.ckks_vector_from(context, dot_data)
# dot_decrypted = dot_vec.decrypt()
# dot_product = sum(dot_decrypted)

# norm1_data = utils.read_data("outputs_manipulator/norm1_squared.txt")
# norm2_data = utils.read_data("outputs_manipulator/norm2_squared.txt")

# norm1_vec = ts.ckks_vector_from(context, norm1_data)
# norm2_vec = ts.ckks_vector_from(context, norm2_data)

# norm1 = math.sqrt(sum(norm1_vec.decrypt()))
# norm2 = math.sqrt(sum(norm2_vec.decrypt()))

# if norm1 != 0 and norm2 != 0:
#     correlation = dot_product / (norm1 * norm2)
# else:
#     correlation = 0.0

# print("\n📊 Résultats déchiffrés :")
# print(f"Distance euclidienne : {distance:.4f}")
# print(f"Corrélation cosinus  : {correlation:.4f}")

# print("\n🧠 Interprétation :")
# if distance < 30: 
#     print(" Les audios sont très similaires (distance faible)")
# else:
#     print(" Les audios semblent différents (distance élevée)")

# if correlation > 0.8:
#     print(" Forte corrélation : les audios sont très similaires")
# elif correlation > 0.5:
#     print(" Corrélation modérée : similarité partielle")
# else:
#     print(" Faible corrélation : audios probablement différents")